# SEC Form D Startup Funding Leads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/sec-form-d-funding-leads.ipynb)

**What:** Startup funding leads from official SEC EDGAR Form D filings at $0.0025 per filing. Issuer, industry, offering size, amount sold, investors, first-sale date, officers. Filter by industry, state, size; exclude pooled funds; dedupe.

**Why:** run the [`sec-form-d-funding-leads`](https://apify.com/transparent_meteorite/sec-form-d-funding-leads) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Phone numbers and related-person (executive/director) data are not displayed; issuer-level fields only.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/sec-form-d-funding-leads) for every option.

In [2]:
run_input = {'lookbackDays': 3, 'excludePooledFunds': True, 'maxResults': 5, 'onlyNewSinceLastRun': False}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/sec-form-d-funding-leads').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> Status: RUNNING, Message: 


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:41.564Z ACTOR: Pulling container image of build 37npZTOajFFDZ4cCP from registry.


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:42.475Z ACTOR: Creating container.


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:42.540Z ACTOR: Starting container.


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:42.541Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:45.634Z Done. Delivered 5 filings (charged 5) from 207 in index; filtered out {"pooled":6}.


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:45.697Z INFO  [Status message]: Done: 5 items


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> 2026-10-13T22:18:45.752Z Thanks for using SEC Form D Funding Leads. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/f49QJumStKuKDXdd8/issues) with the run URL and we will reply there.


[apify.sec-form-d-funding-leads runId:BM9pcAbFrGgceueiu] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

Phone numbers and related-person (executive/director) data are not displayed; issuer-level fields only.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["entityName", "entityType", "issuerState", "filingDate", "industryGroup", "totalOfferingAmount", "totalAmountSold", "numberOfInvestors", "revenueRange"] if c in df.columns]
df[SHOW]

,entityName,entityType,issuerState,filingDate,industryGroup,totalOfferingAmount,totalAmountSold,numberOfInvestors,revenueRange
0,222 Show Ltd Liability Co,Limited Liability Company,NY,2026-10-09,Other,2400000,622000,8,Decline to Disclose
1,8021 Military LLC,Limited Liability Company,TX,2026-10-09,Residential,5000000,0,0,Decline to Disclose
2,AMIRA LEARNING INC,Corporation,CA,2026-10-09,Other,14000000,4999995,3,Not Applicable
3,"AlignAI, LLC",Limited Liability Company,MO,2026-10-09,Other Technology,350000,125000,4,No Revenues
4,American Metal Inc,Corporation,TX,2026-10-09,Other,1600000,1600000,5,Decline to Disclose


## 5. A quick analysis

In [5]:
for c in ['totalOfferingAmount','totalAmountSold','numberOfInvestors']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df['industryGroup'].value_counts())
df.sort_values('totalOfferingAmount',ascending=False)[['entityName','totalOfferingAmount','totalAmountSold']].head(5)

industryGroup
Other               3
Residential         1
Other Technology    1
Name: count, dtype: int64


,entityName,totalOfferingAmount,totalAmountSold
2,AMIRA LEARNING INC,14000000,4999995
1,8021 Military LLC,5000000,0
0,222 Show Ltd Liability Co,2400000,622000
4,American Metal Inc,1600000,1600000
3,"AlignAI, LLC",350000,125000


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [sec-form-d-funding-leads on the Apify Store](https://apify.com/transparent_meteorite/sec-form-d-funding-leads).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.